In [0]:
dbutils.widgets.dropdown(name='Reset', defaultValue='True', choices=['True', 'False'], label="Reset: Delete previous data")


In [0]:
# Create a Unity Catalog Volume
spark.sql("CREATE VOLUME IF NOT EXISTS porya_catalog.default.synthetic_transactions")
# Sets the file path variable 
volume_file_path = "/Volumes/porya_catalog/default/synthetic_transactions/"

In [0]:
# DBTITLE 1,Notebook Variables
nRows = 10
nPositiveRows = round(nRows/3)
destination_path = "{}/no_product_data".format(volume_file_path)
temp_path = "{}/temp".format(volume_file_path)
sleepIntervalSeconds = 1

In [0]:

# DBTITLE 1,Check the Reset Widget Value
if bool(dbutils.widgets.get('Reset')):
  dbutils.fs.rm(temp_path, recurse=True)
  dbutils.fs.rm(destination_path, recurse=True)
  dbutils.fs.mkdirs(destination_path)

# COMMAND ----------

# DBTITLE 1,Data Variables
CustomerID_vars = {"min": 1234, "max": 1260}

Product_vars = {"None": {"min": 1000, "max": 25001, "mean": 15520, "alpha": 4, "beta": 10}}

In [0]:
# %pip install dbldatagen

In [0]:
import dbldatagen as dg
from datetime import datetime
import dbldatagen.distributions as dist
from pyspark.sql.types import IntegerType, FloatType, StringType

def define_specs(Product, Label, currentTimestamp = datetime.now()):
  pVars = Product_vars[Product]
  if Product == "None":
    if Label:
      return (dg.DataGenerator(spark, name="syn_trans", rows=nRows, partitions=4)
        .withColumn("CustomerID", IntegerType(), nullable=False,
                    minValue=CustomerID_vars["min"], maxValue=CustomerID_vars["max"], random=True)
        .withColumn("TransactionTimestamp", "timestamp", 
                    begin=currentTimestamp, end=currentTimestamp,nullable=False,
                  random=False)
        .withColumn("Amount", FloatType(), 
                    minValue=pVars["min"],maxValue=pVars["max"], 
                    distribution=dist.Beta(alpha=pVars["alpha"], beta=pVars["beta"]), random=True)
        .withColumn("Label", IntegerType(), minValue=1, maxValue=1)).build()
    else:
      return (dg.DataGenerator(spark, name="syn_transs", rows=nRows, partitions=4)
        .withColumn("CustomerID", IntegerType(), nullable=False,
                    minValue=CustomerID_vars["min"], maxValue=CustomerID_vars["max"], random=True)
        .withColumn("TransactionTimestamp", "timestamp", 
                    begin=currentTimestamp, end=currentTimestamp,nullable=False,
                  random=False)
        .withColumn("Amount", FloatType(), 
                    minValue=pVars["min"],maxValue=pVars["max"], 
                    distribution=dist.Normal(mean=pVars["mean"], stddev=.001), random=True)
        .withColumn("Label", IntegerType(), minValue=0, maxValue=0)).build()
  else:
    if Label:
      return (dg.DataGenerator(spark, name="syn_trans", rows=nRows, partitions=4)
        .withColumn("CustomerID", IntegerType(), nullable=False,
                    minValue=CustomerID_vars["min"], maxValue=CustomerID_vars["max"], random=True)
        .withColumn("TransactionTimestamp", "timestamp", 
                    begin=currentTimestamp, end=currentTimestamp,nullable=False,
                  random=False)
        .withColumn("Product", StringType(), template=f"Pro\duct \{Product}") 
        .withColumn("Amount", FloatType(), 
                    minValue=pVars["min"],maxValue=pVars["max"], 
                    distribution=dist.Beta(alpha=pVars["alpha"], beta=pVars["beta"]), random=True)
        .withColumn("Label", IntegerType(), minValue=1, maxValue=1)).build()
    else:
      return (dg.DataGenerator(spark, name="syn_transs", rows=nRows, partitions=4)
        .withColumn("CustomerID", IntegerType(), nullable=False,
                    minValue=CustomerID_vars["min"], maxValue=CustomerID_vars["max"], random=True)
        .withColumn("TransactionTimestamp", "timestamp", 
                    begin=currentTimestamp, end=currentTimestamp,nullable=False,
                  random=False)
        .withColumn("Product", StringType(), template=f"Pro\duct \{Product}")
        .withColumn("Amount", FloatType(), 
                    minValue=pVars["min"],maxValue=pVars["max"], 
                    distribution=dist.Normal(mean=pVars["mean"], stddev=.001), random=True)
        .withColumn("Label", IntegerType(), minValue=0, maxValue=0)).build()

# COMMAND ----------

display(define_specs(Product="None", Label=1, currentTimestamp=datetime.now()))

In [0]:
# DBTITLE 1,Functions to generate a JSON dataset for Auto Loader to pick up
from pyspark.sql.functions import expr
from functools import reduce
import pyspark
import os

# Generate a record
def generateRecord(Product,Label):
  return (define_specs(Product=Product, Label=Label, currentTimestamp=datetime.now()))
  

In [0]:
# Generate a list of records
def generateRecordSet():
  Products = ["None"]
  Labels = [0,1]
  recordSet = []
  for Prod in Products:
    for Lab in Labels:
      recordSet.append(generateRecord(Prod, Lab))
  return reduce(pyspark.sql.dataframe.DataFrame.unionByName, recordSet)

In [0]:
def writeJsonFile(destination_path):
  recordDF = generateRecordSet()
  recordDF = recordDF.withColumn("Amount", expr("Amount / 100"))
  recordDF.coalesce(1).write.format("json").save(temp_path)
  
  # Grab the file from the temp location, write it to the location we want and then delete the temp directory
  tempJson = os.path.join(temp_path, dbutils.fs.ls(temp_path)[3][1])
  dbutils.fs.cp(tempJson, destination_path)
  dbutils.fs.rm(temp_path, True)

In [0]:
# Generate a set of data, convert it to a Dataframe, write it out as one json file to the temp path. Then move that file to the destination_path


# DBTITLE 1,Loop for Generating Data
import time

t=1
while(t<5):
  writeJsonFile(destination_path)
  t = t+1
  time.sleep(sleepIntervalSeconds)


# MAGIC %md
# MAGIC #Visually verify
# MAGIC Inspect that we have written records as expected


# DBTITLE 1,Count of Transactions per User
df = spark.read.format("json").load(destination_path)
usercounts = df.groupBy("CustomerID").count()
display(usercounts.orderBy("CustomerID"))



# DBTITLE 1,Display the Data Generated
display(spark.read.format("text").load(destination_path))